# Solar U-Net：团队仓库复现入口
主力为 v2 EfficientNet-B4 U-Net。本 Notebook 已按团队仓库的 `src/`、`configs/`、`analysis/` 布局适配。

默认仅验证文件并展示已有证据，不联网、不安装依赖、不训练、不提交比赛。CPU 缓存重评与 CSV 导出需要另行取得私有运行附件；它们不是重新训练或新图推理。

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, shutil, subprocess, sys
ROOT = next((p for p in [Path.cwd().resolve(), Path.cwd().resolve().parent] if (p/'src/train_seg.py').is_file()), None)
assert ROOT is not None, '请从团队仓库根目录或 notebooks 目录启动。'
RUN_SELF_TESTS = False
RUN_CPU_EVALUATION = False
RUN_TRAINING = False
RUN_EXPORT = False
DATA_PARENT = None  # 官方 MAGFiLO_1.0_Kaggle_2026 文件夹的上级目录。
print('仓库根目录：', ROOT)


## 源码完整性与历史证据

In [ ]:
manifest=json.loads((ROOT/'SOURCE_MANIFEST.json').read_text(encoding='utf-8'))
for name, expected in manifest['sha256'].items():
    assert hashlib.sha256((ROOT/name).read_bytes().replace(b'\r\n',b'\n')).hexdigest()==expected, '文件已修改：'+name
for receipt in json.loads((ROOT/'results/evidence/LIVE_SUBMISSIONS.json').read_text(encoding='utf-8-sig')):
    print(receipt['fileName'], '公榜：', receipt['publicScore'])
oof=json.loads((ROOT/'results/evidence/full_v2_oof/SUMMARY.json').read_text())
print('完整五折研究 OOF：', oof['pq'], '图像：', oof['images'])
for name in ['softlabel_dice_01','softlabel_half_blend_01']:
    result=json.loads((ROOT/'results/evidence'/name/'SUMMARY.json').read_text())
    print(name, '最佳PQ：',result['best']['candidate']['pq'],'相对v2：',result['calibrated_delta'])


## 可选：CPU 自检与完整 OOF 重评分
CPU 依赖：`python -m pip install -r requirements-cpu.txt`。完整 OOF 重评还需要官方训练标注，以及原私有 Notebook 输出中的五个 `probs_f*.npz` 和 `log_f*.json`。

在本地放入 `runs/v2/`（折0/1）、`runs/v2b/`（折2/3/4）。这些文件被团队 `.gitignore` 排除，不提交 GitHub。设置 DATA_PARENT 指向官方数据父目录。重评分不读取测试概率或测试标签。

In [ ]:
if RUN_SELF_TESTS:
    for name in ['test_runtime_guards.py','test_cpu_portability.py']:
        subprocess.run([sys.executable,str(ROOT/'tests'/name)],cwd=ROOT,check=True)
if RUN_CPU_EVALUATION:
    assert DATA_PARENT is not None
    source=Path(DATA_PARENT)/'MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'
    assert source.is_file(), source
    for fold in range(5):
        folder='v2' if fold<2 else 'v2b'
        for ext in [f'probs_f{fold}.npz',f'log_f{fold}.json']:
            assert (ROOT/'runs'/folder/ext).is_file(), '缺少本地运行附件：'+ext
    target=ROOT/'data/MAGFiLO_1.0_Kaggle_2026/train'/source.name
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists():
        assert hashlib.sha256(target.read_bytes()).digest()==hashlib.sha256(source.read_bytes()).digest()
    else:
        shutil.copy2(source,target)
    subprocess.run([sys.executable,str(ROOT/'analysis/verify_full_oof.py'),'--root',str(ROOT)],cwd=ROOT,check=True)


## 可选：从头训练 v2
先在 CUDA 环境安装 `requirements-training.txt`，取得官方完整图像并设置 DATA_PARENT。完整 GPU 版本锁定仍未完成，新的训练也不保证逐位复现历史权重。此项消耗算力；开关默认关闭。

In [ ]:
if RUN_TRAINING:
    import torch
    assert torch.cuda.is_available() and DATA_PARENT is not None
    os.environ['SOLAR_DATA']=str(Path(DATA_PARENT).resolve())
    assert (Path(DATA_PARENT)/'MAGFiLO_1.0_Kaggle_2026/train/train_images').is_dir()
    for config in ['v2.json','v2b.json']:
        subprocess.run([sys.executable,str(ROOT/'src/train_seg.py'),'--config',str(ROOT/'configs'/config),'--out',str(ROOT/'runs/reproduced_v2')],cwd=ROOT,check=True)


## 可选：从冻结概率复现历史 CSV
这一步只读原 v2 的两折测试预测，不读取任何标注、不重新推理或训练，也不会提交 Kaggle。需要提前取回原输出。导出后按忽略换行差异的逐字段哈希核对历史1255行。

In [ ]:
if RUN_EXPORT:
    reference=json.loads((ROOT/'results/evidence/EXPORT_REFERENCE.json').read_text())
    for name,expected in reference['probability_archives'].items():
        assert (ROOT/name).is_file(), '请先取得原私有 Notebook 输出：'+name
        assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==expected, '不是冻结的v2输出：'+name
    sys.path.insert(0,str(ROOT/'src'))
    from make_submission import main as export_csv
    output=ROOT/'outputs/v2_twofold_reproduced.csv'
    output.parent.mkdir(exist_ok=True)
    assert not output.exists(), '请另选新路径，保留已有结果。'
    export_csv({'npz':[str(ROOT/f'runs/v2/probs_f{i}.npz') for i in [0,1]],'cfg':{'t_lo':.4,'close':8,'min_area':300,'min_mean':.85,'min_peak':0},'out':str(output)})
    with output.open(encoding='utf-8-sig',newline='') as f: rows=list(csv.reader(f))
    assert len(rows)-1==reference['rows']==1255
    assert hashlib.sha256(json.dumps(rows,separators=(',',':')).encode()).hexdigest()==reference['normalized_csv_sha256']
    print('历史 CSV 逐字段核对通过；没有提交比赛。')


## 后续工作
当前保留 v2。下一轮 GPU 实验须有区别于已试方向的假设、固定对照及停止条件。补齐最终技术报告、形态示例、IoU/Dice 与分裂/合并统计，以及干净环境中的完整复现验证。

本上传目录不会覆盖团队既有 README、PLAN、LICENSE 或 .gitignore。接入说明见 `docs/INTEGRATION_zh-CN.txt`；方法与来源见 `docs/BASELINE_AND_PROVENANCE.txt`。